# Babel Gauge - Reference Solution

Six legacy systems encode the same six facts differently. Four of the six
encodings are **lossy**: they round temperature/pressure before writing, and
three of them transmit no seconds at all. Two also collapse WARN and FAULT
into a single alarm token.

So this task has a ceiling below 1.0. The work splits in two:

1. **Induce each encoding and invert what is invertible** - field order, units,
   clock convention, date order, status vocabulary, device rendering.
2. **Calibrate what is not** - for a coarse reading, emit the most likely
   underlying value; for a collapsed alarm, infer WARN vs FAULT from context.

Progression below: all-REJECT baseline -> parse-only -> full calibrated solver.

In [ ]:
import collections, json, os, re
import pandas as pd

PUB = "./dataset/public"
if not os.path.isdir(PUB):
    PUB = "./prepared/public"
rd = lambda f: pd.read_csv(os.path.join(PUB, f), keep_default_na=False)

train, test, sites = rd("train.csv"), rd("test.csv"), rd("sites.csv")
off = dict(zip(sites.site, sites.utc_offset_min))
FIELDS = ["ts", "dev", "site", "temp_c", "pres_kpa", "status"]
TY = {"P": "PMP", "C": "CMP", "F": "FAN", "V": "VLV"}
T, DT = pd.Timestamp, pd.Timedelta
print(f"train {len(train):,}   test {len(test):,}   sites {len(sites)}")

## 1. What the six encodings look like

No documentation ships with the data, so start by eyeballing the raw lines.

In [ ]:
for s in train.raw_line.head(400).drop_duplicates().head(8):
    print(s)

## 2. Local scorer

Reproduces the official metric so we can measure each step. Note the
`max(6, len(pieces))` denominator: extra fields dilute the score, missing ones
still divide by 6.

In [ ]:
def line_score(truth, output):
    output = (output or "").strip()
    if truth == "REJECT":
        return 1.0 if output == "REJECT" else 0.0
    if not output or output == "REJECT":
        return 0.0
    pieces = output.split("|")
    got = {}
    for p in pieces:
        if "=" in p:
            k, v = p.split("=", 1)
            got.setdefault(k, v)
    tru = dict(p.split("=", 1) for p in truth.split("|"))
    return sum(1 for k in FIELDS if got.get(k) == tru[k]) / max(6, len(pieces))

def score(preds, answers):
    return sum(line_score(t, o) for t, o in zip(answers, preds)) / len(answers)

# hold out the tail of train to measure without touching the real test set
hold = train.tail(8000)
print("baseline - REJECT everything:",
      round(score(["REJECT"] * len(hold), hold.canonical), 4))

## 3. Induce the encodings

One matcher per family. Everything the line *determines* goes into `known`;
anything it only *constrains* (a rounded reading, a minute-resolution clock, a
collapsed alarm) goes into `coarse` to be calibrated in step 4.

In [ ]:
def observe(raw, off):
    """Parse a raw line into (format, exactly-known fields, coarse readings).

    Anything in `coarse` was rendered at reduced precision, so the exact
    canonical value cannot be derived from it arithmetically.
    """
    m = re.match(r"^(\d{4}-\d\d-\d\d \d\d:\d\d:\d\d) \| ([PCFV]\w\w-\d{4}) @ "
                 r"([A-Z]{2,4}\d) \| T=(-?\d+)F P=(\d+)psi \| (OK|WARN|FAULT)$",
                 raw)
    if m:
        dt, dev, s, t, p, st = m.groups()
        ts = (T(dt.replace(" ", "T")) - DT(minutes=off[s]))
        return "F1", {"ts": ts.strftime("%Y-%m-%dT%H:%M:%SZ"), "dev": dev,
                      "site": s, "status": st}, {"tF": int(t), "pP": int(p)}

    m = re.match(r"^<([PCFV])(\d{4})\|([A-Z]{2,4}\d)\|(\d{14})\|t:(-?[\d.]+)\|"
                 r"p:([\d.]+)\|s:([012])>$", raw)
    if m:
        ty, n, s, dt, t, p, st = m.groups()
        ts = T(f"{dt[:4]}-{dt[4:6]}-{dt[6:8]}T{dt[8:10]}:{dt[10:12]}:{dt[12:]}")
        return "F2", {"ts": ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
                      "dev": f"{TY[ty]}-{n}", "site": s,
                      "temp_c": f"{float(t):.1f}", "pres_kpa": f"{float(p):.1f}",
                      "status": {"0": "OK", "1": "WARN", "2": "FAULT"}[st]}, {}

    if "time=" in raw and "flag=" in raw:
        g = dict(re.findall(r"(\w+)=(\S+)", raw))
        dm = re.search(r"time=(\d\d)/(\d\d)/(\d{4}) (\d\d):(\d\d)", raw)
        if dm and re.match(r"^[PCFV]\w\w\d{4}$", g.get("dev", "")) \
                and g.get("site") in off:
            dd, mm, yy, H, Mi = dm.groups()
            base = T(f"{yy}-{mm}-{dd}T{H}:{Mi}:00") - DT(minutes=off[g["site"]])
            return "F3", {"dev": f"{TY[g['dev'][0]]}-{g['dev'][3:]}",
                          "site": g["site"], "status": g["flag"]}, {
                "minute": base, "tC": int(g["temp"].rstrip("C")),
                "pK": int(g["pressure"].rstrip("kPa"))}

    m = re.match(r"^([PCFV])(\d{4})([A-Z0-9]+)\s+(\d{6})(\d{4})([+-]\d{3})"
                 r"(\d{5})([OA])$", raw)
    if m:
        ty, n, s, dd, hm, t, p, st = m.groups()
        base = T(f"20{dd[:2]}-{dd[2:4]}-{dd[4:6]}T{hm[:2]}:{hm[2:]}:00") \
            - DT(minutes=off[s])
        return "F4", {"dev": f"{TY[ty]}-{n}", "site": s,
                      }, {"minute": base, "tC": int(t), "pK": int(p),
                          "alarm": st == "A"}

    m = re.match(r"^([A-Z]{2,4}\d),([PCFV]\w\w-\d{4}),(\d\d)/(\d\d)/(\d\d) "
                 r"(\d\d):(\d\d)(AM|PM),(-?\d+),(\d+),(RUN|ALM)$", raw)
    if m:
        s, dev, mm, dd, yy, H, Mi, ap, t, p, st = m.groups()
        h = int(H) % 12 + (12 if ap == "PM" else 0)
        base = T(f"20{yy}-{mm}-{dd}T{h:02d}:{Mi}:00") - DT(minutes=off[s])
        return "F5", {"dev": dev, "site": s}, {
            "minute": base, "tF": int(t), "pP": int(p), "alarm": st == "ALM"}

    if raw.startswith("{") and raw.endswith("}"):
        try:
            j = json.loads(raw)
        except ValueError:
            return None, {}, {}
        return "F6", {"ts": T(j["e"], unit="s").strftime("%Y-%m-%dT%H:%M:%SZ"),
                      "dev": j["d"], "site": j["st"],
                      "temp_c": f"{float(j['tc']):.1f}",
                      "pres_kpa": f"{float(j['pk']):.1f}",
                      "status": {"ok": "OK", "warn": "WARN",
                                 "fault": "FAULT"}[j["x"]]}, {}
    return None, {}, {}

## 4. Parse only, no calibration

Invert what is invertible and leave the lossy fields as a direct arithmetic
conversion. This is what a solver gets if it assumes everything is
recoverable.

In [ ]:
def transduce(df, best=None, arule=None, amode="FAULT"):
    out = []
    for raw in df.raw_line:
        fmt, known, coarse = observe(raw, off)
        if fmt is None:
            out.append("REJECT"); continue
        rec = dict(known)
        for key, field in (("tF","temp_c"),("tC","temp_c"),
                           ("pP","pres_kpa"),("pK","pres_kpa")):
            if key in coarse:
                if best is not None:
                    rec[field] = best.get((key, coarse[key]), "0.0")
                elif key == "tF":
                    rec[field] = f"{(coarse[key]-32)*5/9:.1f}"
                elif key == "pP":
                    rec[field] = f"{coarse[key]*6.894757:.1f}"
                else:
                    rec[field] = f"{float(coarse[key]):.1f}"
        if "minute" in coarse:
            rec["ts"] = coarse["minute"].strftime("%Y-%m-%dT%H:%M:%SZ")
        if "alarm" in coarse:
            rec["status"] = ("OK" if not coarse["alarm"]
                             else (arule.get(int(float(coarse.get("tC", coarse.get("tF", 0)))//5), amode)
                                   if arule else amode))
        out.append("|".join(f"{k}={rec.get(k,'')}" for k in FIELDS))
    return out

print("parse only, no calibration:", round(score(transduce(hold), hold.canonical), 4))

## 5. Calibrate the lossy fields

For each coarse reading, learn from the training pairs which canonical value
most often produced it. For the collapsed alarm, WARN vs FAULT is not in the
line - but it tracks temperature, so condition on the reported reading.

In [ ]:
tab = collections.defaultdict(collections.Counter)
alarm = collections.defaultdict(collections.Counter)
fit = train.head(len(train) - len(hold))

for raw, canon in zip(fit.raw_line, fit.canonical):
    if canon == "REJECT":
        continue
    fmt, known, coarse = observe(raw, off)
    if fmt is None:
        continue
    tk = dict(p.split("=", 1) for p in canon.split("|"))
    for key in ("tF", "tC"):
        if key in coarse: tab[(key, coarse[key])][tk["temp_c"]] += 1
    for key in ("pP", "pK"):
        if key in coarse: tab[(key, coarse[key])][tk["pres_kpa"]] += 1
    if "minute" in coarse: tab[("sec", 0)][tk["ts"][-3:-1]] += 1
    if coarse.get("alarm"):
        alarm[int(float(coarse.get("tC", coarse.get("tF", 0))) // 5)][tk["status"]] += 1

best = {k: v.most_common(1)[0][0] for k, v in tab.items()}
arule = {b: c.most_common(1)[0][0] for b, c in alarm.items()}
print("alarm rule by temperature bucket:",
      {b: arule[b] for b in sorted(arule)[:8]}, "...")
print("\ncalibrated solver:",
      round(score(transduce(hold, best, arule), hold.canonical), 4))

## 6. Write the submission

The score above is the ceiling, not a shortfall: the digits those encodings
dropped are not in the input, so no method recovers them.

In [ ]:
os.makedirs("./working", exist_ok=True)
sub = pd.DataFrame({"line_id": test.line_id,
                    "output": transduce(test, best, arule)})
sub.to_csv("./working/submission.csv", index=False)
print(f"wrote {len(sub):,} rows")
sub.head(3)

## A note on the numbers above

The holdout used here is the tail of `train.csv`, and the training split is
richer in the two fully-precise encodings than the test split is. So the
holdout score reads higher than the score this same solver gets on the real
test set (about 0.64). The gap is the documented distribution shift between
splits, not a bug.

Notice also how little the calibration step adds over plain parsing. That is
the point: once an encoding has thrown a digit away, no amount of modelling
brings it back. The score is dominated by how many encodings you invert
correctly, not by how cleverly you guess the missing digits.